# G1 re-run + G2 four-row table

Two stages in one session, in this order for a reason: **G1 runs the full dev split
through the teacher, which populates the cache that G2's rewrite and decompose calls then
hit for free.** Run separately, G2 would pay for those twice.

### Stage 1 — G1 re-run (~1h)

The Phase 1 table has to be re-measured. The verifier prompt existed in two places and
had drifted; unifying them (`core/prompts.py`) changed the inference prompt, which is part
of the LLM cache key. The old numbers — no-retrieval 0.339, plain RAG 0.346, CRAG 0.414 —
came from a prompt that no longer exists in the code, and the C1 results that *do* use the
unified prompt moved by up to 0.13. Those two sets cannot share a table.

### Stage 2 — G2 (~2h)

C4's ExpRAG reproduction: a 2x2 of what the student was trained on against whether it sees
retrieved experience at inference, plus two rows comparing the filtered adapters with the
gold-target ones.

Stage 1 uses `--workers 8` because it is bound on the teacher endpoint. **Stage 2 does
not**: there the student runs locally on the same GPU, so workers would only contend.

In [ ]:
import os
import pathlib
import subprocess
import sys

os.chdir("/kaggle/working")
REPO = pathlib.Path("/kaggle/working/VeriMem")
if not (REPO / ".git").exists():
    subprocess.run(
        ["git", "clone", "-q", "https://github.com/TERABYTE05/VeriMem.git", str(REPO)],
        check=True,
    )
os.chdir(REPO)
sys.path.insert(0, str(REPO))

RESULTS = "/kaggle/working/results"
os.environ["VERIMEM_DATA_DIR"] = "/kaggle/working/data"
os.environ["VERIMEM_CACHE_DIR"] = "/kaggle/working/cache"
os.environ["VERIMEM_RESULTS_DIR"] = RESULTS
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

from core.prompts import prompt_fingerprint

print("verifier prompt:", prompt_fingerprint())

## Inputs and the warm cache

The cache comes from `verimem-c1-adaptive`, which ran under the **unified** prompt — so
its verify calls are reusable here. Restoring a cache from before the unification would
have been worse than useless: every entry would miss, and the restore would just cost
time.

In [ ]:
import pathlib
import shutil

DATA = pathlib.Path("/kaggle/working/data/averitec")
TRAJ = pathlib.Path("/kaggle/working/data/trajectories")
for d in (DATA, TRAJ):
    d.mkdir(parents=True, exist_ok=True)

print("attached:", [p.name for p in pathlib.Path("/kaggle/input").iterdir()])

for name, dest in (
    ("dev.json", DATA), ("pool_dev.jsonl", DATA), ("trajectories.jsonl", TRAJ),
):
    if (dest / name).exists():
        continue
    for found in pathlib.Path("/kaggle/input").rglob(name):
        shutil.copy(found, dest / name)
        break
    else:
        raise SystemExit(f"{name} not found among the attached inputs")
for p in sorted(DATA.iterdir()) + sorted(TRAJ.iterdir()):
    print(f"  {p.name:<24} {p.stat().st_size/1e6:7.1f} MB")

restored = 0
for ns in ("llm", "evaluator"):
    dst = pathlib.Path(f"/kaggle/working/cache/{ns}")
    if dst.exists():
        continue
    for src in pathlib.Path("/kaggle/input").rglob(f"cache/{ns}"):
        if src.is_dir():
            shutil.copytree(src, dst)
            n = len(list(dst.rglob("*.json")))
            restored += n
            print(f"restored {n} cached {ns} calls")
            break
print(f"{restored} cached calls restored" if restored else "WARNING: cold cache")

In [ ]:
!pip install -q transformers datasets accelerate peft bitsandbytes openai pyyaml python-dotenv 2>&1 | tail -2
import torch

print("cuda:", torch.cuda.is_available(), "| bf16:", torch.cuda.is_bf16_supported())

## The evaluator

In [ ]:
def find_evaluator():
    for root in (pathlib.Path("/kaggle/working"), pathlib.Path("/kaggle/input")):
        if not root.exists():
            continue
        for p in root.rglob("model/config.json"):
            if (p.parent / "model.safetensors").exists() or (p.parent / "pytorch_model.bin").exists():
                return p.parent
    return None


EVALUATOR = find_evaluator()
if EVALUATOR is None:
    raise SystemExit("no evaluator attached - attach the verimem-g1 kernel output")
print("EVALUATOR:", EVALUATOR)

from evaluator.score import T5Scorer

scorer = T5Scorer(EVALUATOR)
s = scorer.score(
    "The Kochi metro opened to the public in 2017.",
    ["The Kochi Metro began commercial operations in June 2017 after several delays.",
     "Penguins are flightless birds found mainly in the southern hemisphere."],
)
print(f"relevant {s[0]:+.3f}   irrelevant {s[1]:+.3f}")
assert s[0] > s[1], "evaluator is not separating - check the model"

## The adapters

A missing gold adapter stops the run: the filtered and gold-target adapters were trained
on different label distributions (5 vs 106 `Conflicting evidence` examples), so
substituting one for the other would report a table that was never run.

In [ ]:
ADAPTERS = {}
for root in (pathlib.Path("/kaggle/working"), pathlib.Path("/kaggle/input")):
    if not root.exists():
        continue
    for cfg in root.rglob("adapter/adapter_config.json"):
        run = cfg.parent.parent.name
        ADAPTERS.setdefault(run.split("_", 1)[-1] if "_" in run else run, cfg.parent)

for k in sorted(ADAPTERS):
    size = sum(f.stat().st_size for f in ADAPTERS[k].rglob("*") if f.is_file())
    print(f"  {k:<18} {size/1e6:7.1f} MB  {ADAPTERS[k]}")

NEEDED = ("lora-sft-gold", "rag-sft-gold")
missing = [k for k in NEEDED if k not in ADAPTERS]
if missing:
    raise SystemExit(f"missing adapters {missing} - attach the verimem-sft-gold output")
OPTIONAL = [k for k in ("lora-sft", "rag-sft") if k in ADAPTERS]
print(f"\nrequired: {list(NEEDED)}  |  filtered comparison: {OPTIONAL or 'not attached, skipped'}")

## Teacher endpoint

In [ ]:
import shutil
import subprocess
import time
import urllib.request

# Both ends have to allow concurrency, and these must be set before `ollama serve`
# starts or the server inherits the defaults. 8 fits a 16 GB T4 beside the evaluator.
os.environ["OLLAMA_NUM_PARALLEL"] = "8"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"

if shutil.which("zstd") is None:
    subprocess.run("apt-get -qq update && apt-get -qq install -y zstd",
                   shell=True, capture_output=True)
if shutil.which("ollama") is None:
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True, capture_output=True, text=True)
OLLAMA_BIN = shutil.which("ollama")
assert OLLAMA_BIN, "ollama did not install"

try:
    urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
except Exception:
    subprocess.Popen([OLLAMA_BIN, "serve"], stdout=subprocess.DEVNULL,
                     stderr=subprocess.DEVNULL, start_new_session=True)
    for _ in range(120):
        try:
            urllib.request.urlopen("http://localhost:11434/api/tags", timeout=2)
            break
        except Exception:
            time.sleep(1)
    else:
        raise RuntimeError("ollama did not start")
subprocess.run([OLLAMA_BIN, "pull", "qwen2.5:7b"], check=True)
print("teacher up", flush=True)

os.environ["VERIMEM_API_BASE_URL"] = "http://localhost:11434/v1"
os.environ["VERIMEM_API_MODEL"] = "qwen2.5:7b"
os.environ.setdefault("VERIMEM_API_KEY", "ollama")

DEV = "/kaggle/working/data/averitec/dev.json"
POOL = "/kaggle/working/data/averitec/pool_dev.jsonl"
TRAJECTORIES = "/kaggle/working/data/trajectories/trajectories.jsonl"

## Stage 1 — G1 re-run, whole dev split, three systems

`--workers 8`: this stage is bound on the teacher. It also warms the cache that stage 2
reads.

In [ ]:
r = subprocess.run(
    [sys.executable, "-u", "-m", "eval.run_baselines",
     "--data", DEV, "--store", POOL, "--evaluator", str(EVALUATOR),
     "--system", "all", "--workers", "8", "--yes", "--name", "g1-rerun"],
    env=os.environ,
)
if r.returncode != 0:
    raise SystemExit("G1 re-run failed")

## Stage 2 — G2, six rows

No `--workers`: the student runs locally on this GPU, so concurrency would contend rather
than help. The teacher calls these rows need are already cached by stage 1.

In [ ]:
ROWS = [
    # (name, adapter key or None, retrieval at inference)
    ("g2-prompted-3b",            None,            False),
    ("g2-prompted-3b-retrieval",  None,            True),
    ("g2-lora-sft",               "lora-sft-gold", False),
    ("g2-rag-sft",                "rag-sft-gold",  True),
]
for key in OPTIONAL:
    ROWS.append((f"g2-{key}-filtered", key, key.startswith("rag")))

LIMIT = "300"
for name, key, retrieval in ROWS:
    argv = [sys.executable, "-u", "-m", "eval.run_baselines",
            "--data", DEV, "--store", POOL, "--evaluator", str(EVALUATOR),
            "--system", "crag", "--limit", LIMIT, "--yes", "--name", name]
    argv += ["--verifier-base"] if key is None else ["--verifier-adapter", str(ADAPTERS[key])]
    if retrieval:
        argv += ["--trajectories", TRAJECTORIES, "--experience-retrieval"]
    print(f"\n######## {name}  adapter={key or 'none'}  retrieval={retrieval} ########",
          flush=True)
    r = subprocess.run(argv, env=os.environ)
    if r.returncode != 0:
        raise SystemExit(f"{name} failed")

## Both tables

In [ ]:
import json
import pathlib

LABELS = ["Supported", "Refuted", "Not enough evidence", "Conflicting evidence"]
found = {}
for f in sorted(pathlib.Path(RESULTS).rglob("metrics.json")):
    found[f.parent.name.split("_", 1)[-1]] = json.loads(f.read_text())

print("=== Stage 1: G1 re-run (whole dev split, unified prompt) ===")
g1 = found.get("g1-rerun", {})
print(f"  {'system':<16}{'macro-F1':>10}{'accuracy':>10}{'n':>7}")
for sysname in ("no_retrieval", "plain_rag", "crag"):
    m = g1.get(sysname)
    if m:
        print(f"  {sysname:<16}{m['macro_f1']:>10.3f}{m['accuracy']:>10.3f}{m['n']:>7}")
print("\n  for comparison, the superseded pre-unification numbers were")
print("  no_retrieval 0.339 | plain_rag 0.346 | crag 0.414")

print("\n=== Stage 2: G2 four-row table (+ filtered comparison) ===")
print(f"{'row':<26}{'macro-F1':>10}{'accuracy':>10}" + "".join(f"{label[:12]:>14}" for label in LABELS))
print("-" * (46 + 14 * len(LABELS)))
for name, _, _ in ROWS:
    d = found.get(name)
    if not d:
        print(f"{name:<26}{'(missing)':>10}")
        continue
    m = d.get("crag", d)
    per = m.get("per_class", {})
    line = f"{name:<26}{m.get('macro_f1', 0):>10.3f}{m.get('accuracy', 0):>10.3f}"
    for label in LABELS:
        v = per.get(label, {})
        f1 = v.get("f1") if isinstance(v, dict) else v
        line += f"{'-' if f1 is None else format(f1, '.3f'):>14}"
    print(line)

print("\nExpRAG's claim, as a comparison:")
try:
    g = {k: found[k].get("crag", found[k])["macro_f1"] for k in
         ("g2-prompted-3b", "g2-prompted-3b-retrieval", "g2-lora-sft", "g2-rag-sft")}
    retrieval_alone = g["g2-prompted-3b-retrieval"] - g["g2-prompted-3b"]
    training_alone = g["g2-lora-sft"] - g["g2-prompted-3b"]
    both = g["g2-rag-sft"] - g["g2-prompted-3b"]
    print(f"  retrieval alone    {retrieval_alone:+.3f}")
    print(f"  training alone     {training_alone:+.3f}")
    print(f"  both together      {both:+.3f}")
    print(f"  super-additive?    {both - retrieval_alone - training_alone:+.3f}  (> 0 supports ExpRAG)")
except KeyError as exc:
    print(f"  incomplete: {exc}")

In [ ]:
import shutil

shutil.rmtree("/kaggle/working/VeriMem/.git", ignore_errors=True)
for p in pathlib.Path(RESULTS).rglob("checkpoints"):
    shutil.rmtree(p, ignore_errors=True)
!du -sh /kaggle/working/cache /kaggle/working/results